# MI-BCI-FPGA — Version 4: second dataset (PhysioNet, 104 subjects) + online-EA analysis

**Inputs:** the new code dataset (hanet_mi.zip) **and** the output of MI-BCI-FPGA Version 3. Accelerator: GPU T4×2, Internet ON. Then *Save & Run All*.

In [ ]:
# 1. Setup: find code, install MOABB, restore previous outputs (resume)
import os, glob, shutil, subprocess, sys, time, zipfile
T0 = time.time()
LIMIT_H = float(os.environ.get('HANET_LIMIT_H', 11.0))   # stop starting new runs after this many hours
IN, BASE = os.environ.get('HANET_IN', '/kaggle/input'), os.environ.get('HANET_BASE', '/kaggle/working')
CODE = f'{BASE}/hanet_mi/python'

# --- code: from the uploaded dataset (extracted folder or zip) ---
if not os.path.exists(CODE):
    src = glob.glob(f'{IN}/**/hanet_mi/python', recursive=True)
    if src:
        shutil.copytree(os.path.dirname(src[0]), f'{BASE}/hanet_mi')
    else:
        z = glob.glob(f'{IN}/**/hanet_mi*.zip', recursive=True)
        assert z, 'Upload hanet_mi.zip as a Kaggle dataset and add it as input'
        zipfile.ZipFile(z[0]).extractall(BASE)
os.chdir(CODE); sys.path.insert(0, CODE)
print('code at', CODE)

# --- resume: copy runs/cache/pretrain/mne_data from attached previous notebook outputs ---
# Kaggle may mount an attached output several levels deep (e.g. /kaggle/input/notebooks/<user>/<name>/),
# so search up to 4 levels and accept a folder only if it sits next to a log.txt or runs/ folder.
print('inputs:', [p for p in glob.glob(f'{IN}/*/*/*') if os.path.isdir(p)][:20])
for d in ('runs', 'cache', 'pretrain'):   # mne_data skipped: thousands of EEG files, very slow to copy
    for depth in range(1, 5):
        for prev in glob.glob(f'{IN}/' + '*/' * depth + d):
            root = os.path.dirname(prev)
            if not (os.path.exists(f'{root}/log.txt') or os.path.isdir(f'{root}/runs')):
                continue
            shutil.copytree(prev, f'{BASE}/{d}', dirs_exist_ok=True); print('restored', prev)
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
print('restored runs:', sorted(os.listdir(f'{BASE}/runs')))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'moabb'], check=False)
os.environ['MNE_DATA'] = f'{BASE}/mne_data'
import mne; mne.set_config('MNE_DATA', f'{BASE}/mne_data', set_env=True)
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

print('cache files:', sorted(os.listdir(f'{BASE}/cache')))


In [ ]:
# 2. Helpers
PROP = '--no-attention --K1 64 --P1 4 --P2 8 --act elu --dg none'
ATTN = '--K1 64 --P1 4 --P2 8 --act elu --dg none'
EEG, SHAL = '--model eegnet --dg none', '--model shallow --dg none'
CACHE = f'--cache-dir {BASE}/cache'
FT = '--epochs 80 --patience 20'          # fine-tuning from pretrained weights converges faster
LOG = open(f'{BASE}/log.txt', 'a')

def hours(): return (time.time() - T0) / 3600

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for l in p.stdout:
        if l.startswith(('S0', 'F0', 'F1', 'F2', 'F3', 'F4', 'k=', '  k=', '  K=', '  eps', 'oracle', 'warm-up', '  ', '[init]', '[data]', '[pretrain]', 'saved', 'loaded', 'train ', 'ep ',
                         'target', 'fold', 'exported', 'Traceback')) or 'error' in l.lower():
            print(l, end='', flush=True); LOG.write(l); LOG.flush()
    p.wait()

def run(name, extra, dataset='2a', subjects=range(1, 10)):
    for s in subjects:
        out = f'{BASE}/runs/{name}/S{s}'
        if os.path.exists(f'{out}/results.json'):
            continue
        if hours() > LIMIT_H:
            print(f'** time limit reached, {name} S{s} left for next session'); return
        sh(f'{sys.executable} -W ignore train_loso.py --dataset {dataset} --targets {s} --out {out} {CACHE} {extra}')
    print(f'== {name} done ({hours():.2f} h)', flush=True)

def pretrain(name, extra, classes='all'):
    out = f'{BASE}/pretrain/{name}.pt'
    if not os.path.exists(out) and hours() < LIMIT_H:
        sh(f'{sys.executable} -W ignore pretrain.py --out {out} --classes {classes} {CACHE} {extra}')
    return out

def table(groups, metric='online_ea', csv=None):
    g = ' '.join(f'--group "{n}={r}"' for n, r in groups)
    c = f'--csv {csv}' if csv else ''
    sh_all = subprocess.run(f'{sys.executable} aggregate.py --base {BASE}/runs --metric {metric} {g} {c}',
                            shell=True, capture_output=True, text=True)
    print(sh_all.stdout + sh_all.stderr); LOG.write(sh_all.stdout); LOG.flush()

def seeds(prefix): return ','.join(f'{prefix}_s{s}' for s in (42, 7, 123))

def run_kfold(name, extra, dataset='physionet', folds=5):
    for f in range(folds):
        out = f'{BASE}/runs/{name}/F{f}'
        if os.path.exists(f'{out}/results.json'):
            continue
        if hours() > LIMIT_H:
            print(f'** time limit reached, {name} F{f} left for next session'); return
        sh(f'{sys.executable} -W ignore train_loso.py --dataset {dataset} --protocol kfold --folds {folds} '
           f'--fold {f} --out {out} {CACHE} {extra}')
    print(f'== {name} done ({hours():.2f} h)', flush=True)


In [ ]:
# 3. Test data for the FPGA board: subject 1 of IV-2a, filtered EEG (not aligned), int32 Q.10
import numpy as np
f = glob.glob(f'{BASE}/cache/hanet_cache_2a_all_all_v2.npz')[0]
z = np.load(f, allow_pickle=True)
X, y, s = z['X1'].astype(np.float64), z['y1'], z['s1']
_, sess = np.unique(s, return_inverse=True)
Xq = np.clip(np.round(X * 1024), -2**23, 2**23 - 1).astype(np.int32)
np.savez_compressed(f'{BASE}/board_S1.npz', X_q10=Xq, y=y.astype(np.int32), session=sess.astype(np.int32))
print('board_S1.npz', Xq.shape, 'classes', np.bincount(y), 'sessions', np.bincount(sess))


In [ ]:
# 4. Online-EA analysis on the trained IV-2a models (inference only, ~10 min):
#    warm-up curve + sensitivity to k (forgetting), K (Newton-Schulz iterations), eps (shrinkage)
r = subprocess.run(f'{sys.executable} -W ignore analyze_online_ea.py --runs {BASE}/runs/A4_prop_s42 '
                   f'--cache-dir {BASE}/cache --out {BASE}/online_ea_analysis.json',
                   shell=True, capture_output=True, text=True)
print(r.stdout[-4000:] + r.stderr[-2000:]); LOG.write(r.stdout); LOG.flush()


In [ ]:
# 5. Second dataset: PhysioNet MI, 104 subjects, same 22 channels / preprocessing / model,
#    subject-exclusive 5-fold CV (target subjects never seen, no labels, online EA per subject)
PN = '--epochs 100 --patience 20'
for sd in (42, 7, 123):
    run_kfold(f'P4_prop_s{sd}',   f'{PROP} {PN} --seed {sd}')
    run_kfold(f'P4_eegnet_s{sd}', f'{EEG} {PN} --seed {sd}')
for sd in (42, 7, 123):          # 2-class: left vs right hand
    run_kfold(f'P2_prop_s{sd}',   f'{PROP} {PN} --classes lr --seed {sd}')
    run_kfold(f'P2_eegnet_s{sd}', f'{EEG} {PN} --classes lr --seed {sd}')
print(f'QUEUE FINISHED in {hours():.2f} h')


In [ ]:
# 6. Tables
for m in ('online_ea', 'offline_ea_oracle', 'no_ea'):
    table([('Prop', seeds('P4_prop')), ('EEGNet', seeds('P4_eegnet'))], m, f'{BASE}/tableP4_{m}.csv')
    table([('Prop', seeds('P2_prop')), ('EEGNet', seeds('P2_eegnet'))], m, f'{BASE}/tableP2_{m}.csv')
shutil.rmtree(f'{BASE}/hanet_mi', ignore_errors=True)
print('ALL DONE', f'{hours():.2f} h')
